In [49]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage


In [6]:
load_dotenv(override=True)

True

In [30]:
loader = PyPDFLoader("CV -  Alex CHIMI - 20260204.pdf")

In [31]:
tokennizer = tiktoken.encoding_for_model("gpt-4o-mini")


In [32]:
print(tokennizer.name)

o200k_base


In [33]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name= tokennizer.name,
    chunk_size = 300,
    chunk_overlap = 20
)

In [34]:
chunks = loader.load_and_split(splitter)

In [35]:
print(len(chunks))

11


In [36]:
print(chunks[0].metadata)

{'producer': 'Microsoft® PowerPoint® 2019', 'creator': 'Microsoft® PowerPoint® 2019', 'creationdate': '2026-02-03T03:41:31+01:00', 'author': 'Alex Bleriot Chimi Nantchouang', 'moddate': '2026-02-03T03:41:31+01:00', 'source': 'CV -  Alex CHIMI - 20260204.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [37]:
print(chunks[0].page_content)

Data
Foundation
1
Compétences principales.
Big Data - Data Engineering 
Spark  (Certified),
Snowflake( Certfied Advanced Data engineer),
Snowpark,
Control-m,
Databricks (certified),
Hadoop, Azure, Hive, SQL,
Spark Structured Streaming,
Datastage
Intégration continue 
Git, GitHub, Jenkins,  Github Actions
Organisation et méthode
Travail en équipe, méthode agile, montée en compétence rapide
Expériences professionnelles.
Alex 
CHIMI
Data Engineer
Diplômes & Certifications.
opérationnel
Langues.
courant
Français Anglais
Ingénieur Big data  et Machine 
Learning
EFREI
promotion  2021
Databricks Certified Developer 
Associate for Spark 3.0
Snowflake Certified 
Snow Pro core
Data Engineer - Société Générale
03/2024 –Aujourd’hui
Contexte 1:  
Dans le cadre de la migration des plateformes Cloudera et Teradata vers 
Snowflake, Société Générale a lancé un projet stratégique baptisé OLAF
(On-premise to Lakehouse migration for Analytics Foundations).Ce projet 
vise à moderniser les traitements analy

In [38]:
embedding_model = OpenAIEmbeddings()

In [39]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="cv_data_collection"
)

In [40]:
retriever = vector_store.as_retriever(kwargs={'k':10})

In [43]:
@tool
def retriever_tool(query : str) -> str:
    """
     Permet de chercher des informations sur des candidats:
     - Nom, Prénom, Diplômes
     - Expériences
     - Competences
    """
    relevant_chunks = retriever.invoke(query)
    context_list = [d.page_content for d in relevant_chunks]
    context = ". ".join(context_list)
    return context
    

In [60]:
@tool
def get_company_infos(company_name :str):
    """
    Consulter des informations sur l'entreprise donnée
    """
    return {
        "company_name" : company_name,
        "domain" : "IT",
        "turnover": 120_870_000
    }

In [61]:
## creer l'agent qui a un RAG comme tools d'abord avec langchain
# agent il faut lui donner le cerveau: llm
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    model=llm,
    tools=[retriever_tool, get_company_infos], # c'est lui qui permet d'exploiter le RAG
    system_prompt="Répond à la question de l'utilisateur en utilisant les tools fournis"
)


In [63]:
resp = agent.invoke(input={
    "messages":[
        HumanMessage("Nom,Prenom, diplômes de Alex CHIMI et les informations sur son entreprise")
    ]
})

In [64]:
print(resp['messages'][-1].content)

### Informations sur Alex CHIMI

- **Nom**: Alex CHIMI
- **Diplômes**:
  - Ingénieur Big Data et Machine Learning, EFREI, promotion 2021
  - Databricks Certified Developer Associate for Spark 3.0
  - Snowflake Certified Snow Pro Core

- **Langues**: 
  - Français (courant)
  - Anglais (courant)

- **Expériences professionnelles**:
  - **Data Engineer – Société Générale** (03/2024 – Aujourd’hui)
    - Projet OLAF : Migration des plateformes Cloudera et Teradata vers Snowflake.
  - **Data Engineer – EDF** (02/2023 – 10/2023)
    - Migration de ScopeGP vers ITAM au sein du Datalake IT de EDF.
  - **Data Engineer – ST Microelectronics** (03/2021 – 09/2021)
    - Programme Manufacturing Data and Analytics pour optimiser la production.
  - **Data Engineer – Société Générale** (10/2021 – 10/2022)
    - Construction et industrialisation de tableaux de bord décisionnels.

- **Compétences principales**:
  - Big Data, Data Engineering, Spark, Snowflake, Hadoop, Azure, SQL, Git, Jenkins, etc.

###

In [65]:
from IPython.display import Markdown

In [66]:
print(display(Markdown(resp['messages'][-1].content)))

### Informations sur Alex CHIMI

- **Nom**: Alex CHIMI
- **Diplômes**:
  - Ingénieur Big Data et Machine Learning, EFREI, promotion 2021
  - Databricks Certified Developer Associate for Spark 3.0
  - Snowflake Certified Snow Pro Core

- **Langues**: 
  - Français (courant)
  - Anglais (courant)

- **Expériences professionnelles**:
  - **Data Engineer – Société Générale** (03/2024 – Aujourd’hui)
    - Projet OLAF : Migration des plateformes Cloudera et Teradata vers Snowflake.
  - **Data Engineer – EDF** (02/2023 – 10/2023)
    - Migration de ScopeGP vers ITAM au sein du Datalake IT de EDF.
  - **Data Engineer – ST Microelectronics** (03/2021 – 09/2021)
    - Programme Manufacturing Data and Analytics pour optimiser la production.
  - **Data Engineer – Société Générale** (10/2021 – 10/2022)
    - Construction et industrialisation de tableaux de bord décisionnels.

- **Compétences principales**:
  - Big Data, Data Engineering, Spark, Snowflake, Hadoop, Azure, SQL, Git, Jenkins, etc.

### Informations sur l'entreprise d'Alex CHIMI

- **Nom de l'entreprise**: Alex CHIMI
- **Domaine**: IT
- **Chiffre d'affaires**: 120,870,000 € 

Si vous avez besoin de plus d'informations, n'hésitez pas à demander !

None
